# Cincinnati Reds: What Actually Brings Fans to the Stadium?

### Author: Artemio BRESCIANI


**Data:** 80 home games, Cincinnati Reds, one MLB season
**Goal:** Figure out which factors really drive paid attendance, using a multiple linear regression model (OLS).

Quick overview of what's in this notebook:

1. Setup and data loading
2. Exploratory analysis (get to know the data)
3. Step 1: Multicollinearity check + single-variable regressions
4. Step 2: Best subset search across all 31 possible models
5. Step 3: The best model, fully interpreted
6. Step 4: Checking that the model is statistically valid (residual analysis)
7. Step 5: Testing square-root transformations  -  do they help?
8. 2D visualisations
9. 3D interactive visualisations
10. Conclusions and business recommendations
11. Appendix A: Extended exploratory analyses (original work preserved)
12. Appendix B: Earlier draft sections (preserved for reference)

> Place `G4 excel dataset.csv` in the same folder as this notebook before running.


<a id="setup"></a>

## 0. Setup

Install and import everything needed.

In [ ]:
%pip install pandas numpy statsmodels scipy matplotlib seaborn scikit-learn plotly networkx


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import itertools
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde, jarque_bera, probplot
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.diagnostic import het_breuschpagan
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

np.random.seed(42)

# Colour palette based on the rocket colormap
ROCKET = sns.color_palette("rocket", as_cmap=True)
rocket_list = sns.color_palette("rocket", 8)

# Named colours pulled from the rocket palette
C_DARK   = rocket_list[0]    # very dark purple-red
C_MID    = rocket_list[3]    # mid orange
C_LIGHT  = rocket_list[6]    # pale pink-cream
C_RED    = "#FF2A2A"         # Ferrari red for best-model highlights
C_ACCENT = rocket_list[5]

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("We are ready to start.")


<a id="data"></a>

## 1. Loading the Data

The raw CSV file sometimes has junk rows at the bottom (summary rows added by Excel). To handle this cleanly, we use `pd.to_numeric(errors="coerce")`, which silently turns any non-numeric value into `NaN`, and then we drop those rows with `dropna()`.

That's the only cleaning we do. No observations are changed or imputed  -  we either keep them as-is or drop them if they're corrupted.

A few notes on the variables:
- `Win%` and `OpWin%` are stored as integers scaled by 1000 (e.g., 407 means a 0.407 win rate). We keep this scale throughout  -  the regression coefficients will reflect it.
- `Weekend` and `Promotion` are already 0/1 dummy variables. No recoding needed.


In [ ]:
# Robust CSV path resolution: handle both common filenames (with space or underscore)
CANDIDATES = ["G4_excel_dataset.csv", "G4 excel dataset.csv"]
CSV_PATH = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])

df_raw = pd.read_csv(CSV_PATH, sep=None, engine="python")

# Normalize columns to a known canonical form using a case-insensitive map
canon = {c.strip().upper(): c for c in df_raw.columns}
rename_map = {}
for src_upper, target in [
    ("ATTENDANCE", "Attendance"),
    ("TEMP",       "Temp"),
    ("WIN%",       "Win%"),
    ("OPWIN%",     "OpWin%"),
    ("WEEKEND",    "Weekend"),
    ("PROMOTION",  "Promotion"),
]:
    if src_upper in canon:
        rename_map[canon[src_upper]] = target

df_raw = df_raw.rename(columns=rename_map)

COLS = ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]
df = df_raw[COLS].copy()
for c in COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna().reset_index(drop=True)

print(f"Clean observations loaded: {len(df)}")
print(f"Any missing values left? {df.isnull().sum().sum()}")
print(f"Any duplicate rows? {df.duplicated().sum()}")
print(f"Attendance range: {df['Attendance'].min():,} to {df['Attendance'].max():,}")
print(f"Weekend breakdown: {df['Weekend'].value_counts().to_dict()}")
print(f"Promotion breakdown: {df['Promotion'].value_counts().to_dict()}")
print()
df.describe().round(2)


<a id="eda"></a>

## 2. Getting to Know the Data (EDA)

Before running any model, it's worth spending a bit of time just looking at the data. Three things we want to check:

1. **How is attendance distributed?** Is it roughly normal, or is it very skewed? This affects how we interpret the model later.
2. **How correlated are the variables with each other?** If two predictors are very correlated, including both of them in the same model can cause problems (multicollinearity).
3. **Do the dummy variables visually shift attendance?** A quick box plot can tell us whether weekend games or promo days obviously attract more fans, before we even run the regression.


In [ ]:
import matplotlib.colors as mcolors
from scipy.stats import probplot

# Use the canonical column name from our cleaned dataframe
attendance_data = df["Attendance"]

# Calculation of normalization parameters
mean_val   = attendance_data.mean()
median_val = attendance_data.median()
vmin       = attendance_data.min()
vmax       = attendance_data.max()

# Build a balanced norm centred on the mean
norm = mcolors.TwoSlopeNorm(vcenter=mean_val, vmin=vmin, vmax=vmax)
icefire_cmap = sns.color_palette("icefire", as_cmap=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Attendance Distribution - Centered Icefire Mapping", fontweight="bold", fontsize=14)

# Histogram with calibrated spectrum
n_bins = 20
counts, bins, patches = axes[0].hist(attendance_data, bins=n_bins, edgecolor="white", alpha=0.9)

bin_centers = 0.5 * (bins[:-1] + bins[1:])
for center, patch in zip(bin_centers, patches):
    color = icefire_cmap(norm(center))
    patch.set_facecolor(color)

# Central tendency indicators
axes[0].axvline(mean_val,   color="black",   lw=2, label=f"Mean (Center): {mean_val:,.0f}")
axes[0].axvline(median_val, color="#333333", lw=2, linestyle="--", label=f"Median: {median_val:,.0f}")

axes[0].set_xlabel("Attendance")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram: Color Shift at Mean")
axes[0].legend()

# Coordinated Q-Q plot
res = probplot(attendance_data, dist="norm", plot=axes[1])
axes[1].get_lines()[0].set(markerfacecolor="#002d5c", markeredgecolor="white", alpha=0.5, markersize=6)
axes[1].get_lines()[1].set(color="#a50021", lw=2.5)
axes[1].set_title("Q-Q Plot: Normality Diagnostic")

plt.tight_layout()
plt.show()

print("Palette Calibration:")
print(f" - White point (vcenter): {mean_val:,.2f}")
print(f" - Asymmetric range: {vmin:,.0f} (Blue) <---> {vmax:,.0f} (Red)")


In [ ]:
# Correlation matrix using the canonical column names (no global mutation of df)
features = ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]
existing_features = [f for f in features if f in df.columns]
if not existing_features:
    raise ValueError("None of the specified features were found in the DataFrame.")

corr = df[existing_features].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)

ax.set_title("Pearson Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

# Print specific pairwise correlations
for pair in [("Weekend", "Attendance"), ("OpWin%", "Attendance"), ("Promotion", "Attendance")]:
    if pair[0] in corr.index and pair[1] in corr.columns:
        print(f"{pair[0]} vs {pair[1]}: r = {corr.loc[pair[0], pair[1]]:.3f}")


In [ ]:
# Box plots: does Weekend / Promotion visually shift attendance?
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Attendance by Category", fontweight="bold")

rocket_two = sns.color_palette("rocket", 5)

for ax, var, labels, col_pair in zip(
        axes,
        ["Weekend", "Promotion"],
        [["Weekday", "Weekend"], ["No Promo", "Promotion Day"]],
        [[rocket_two[1], rocket_two[3]], [rocket_two[1], rocket_two[4]]]):

    groups = [df.loc[df[var] == v, "Attendance"].values for v in [0, 1]]
    bp = ax.boxplot(groups, patch_artist=True, widths=0.45,
                    medianprops=dict(color="white", lw=2.5))
    for patch, col in zip(bp["boxes"], col_pair):
        patch.set_facecolor(col)
        patch.set_alpha(0.85)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Attendance")
    ax.set_title(f"Effect of {var}")

    for i, g in enumerate(groups, 1):
        ax.text(i, g.max() + 500, f"mean={g.mean():,.0f}\nn={len(g)}",
                ha="center", fontsize=8, color="#333")

    t_stat, p_val = stats.ttest_ind(groups[0], groups[1])
    ax.set_xlabel(f"t-test p-value = {p_val:.4f} "
                  f"({'significant' if p_val < 0.05 else 'not significant'})")

plt.tight_layout()
plt.show()


<a id="step1a"></a>

## Step 1  -  Multicollinearity Check (VIF) Before Anything Else

### What is the VIF and why does it matter?

Before running multi-variable regressions, we need to check whether any two predictors are too correlated with each other. If they are, including both of them can make the individual coefficients unreliable (even if the model as a whole looks fine).

The tool we use for this is the **Variance Inflation Factor (VIF)**. It works like this: for each predictor, we temporarily treat it as the dependent variable and regress it against all the other predictors. The higher the R-squared of that regression, the more "explainable" this predictor is by the others, meaning it carries redundant information.

The formula is:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

**Rule of thumb:** if VIF > 5, that variable is a candidate for removal. If VIF > 10, it's a serious problem.

We also run single-variable regressions just to get a baseline feel for each predictor on its own.


In [ ]:
y      = df["Attendance"]
x_cols = ["Temp", "Win%", "OpWin%", "Weekend", "Promotion"]

X_full = sm.add_constant(df[x_cols])
vif_df = pd.DataFrame({
    "Variable": x_cols,
    "VIF": [variance_inflation_factor(X_full.values, i + 1) for i in range(len(x_cols))],
})

print("Variance Inflation Factors")
print("-" * 35)
print(vif_df.to_string(index=False))
print()

any_above_5 = vif_df[vif_df["VIF"] > 5]
if any_above_5.empty:
    print("All VIF values are below 5.")
    print("No multicollinearity issues. All five variables are kept for the subset search.")
else:
    print(f"Warning: {any_above_5['Variable'].tolist()} are above the threshold.")


In [ ]:
# Single-variable regressions - just to see each predictor on its own
print(f"Single-variable OLS results (n = {len(df)})")
print("-" * 70)

single_rows = []
for v in x_cols:
    m  = sm.OLS(y, sm.add_constant(df[[v]])).fit()
    pv = m.pvalues[v]
    single_rows.append({
        "Variable"   : v,
        "Coefficient": round(m.params[v], 4),
        "Std Error"  : round(m.bse[v], 4),
        "t-stat"     : round(m.tvalues[v], 3),
        "p-value"    : round(pv, 4),
        "R squared"  : round(m.rsquared, 4),
        "Adj R sq"   : round(m.rsquared_adj, 4),
        "Significant": "yes" if pv < 0.05 else "no",
    })

single_df = pd.DataFrame(single_rows)
print(single_df.to_string(index=False))
print()
print("Weekend, OpWin%, and Promotion are individually significant (p < 0.05).")
print("Temp and Win% are not significant on their own.")


<a id="step1b"></a>

## Step 2  -  Testing All 31 Possible Models

### How does best subset selection work?

With 5 candidate predictors, the total number of non-empty subsets we can form is:

$$2^5 - 1 = 31 \text{ models}$$

We fit every one of them and rank by **Adjusted R-squared**, which is a corrected version of R-squared that penalises you for adding variables that don't really help:

$$\bar{R}^2 = 1 - (1 - R^2)\frac{n-1}{n-k-1}$$

where n is the number of observations (80) and k is the number of predictors in the model. The key thing is that adding a useless variable will *lower* the Adjusted R-squared, so we can safely compare models with different numbers of predictors.

We also flag whether **all p-values in a model are below 0.05**  -  because a good model should not include variables that aren't statistically contributing.


In [ ]:
all_models = []

for r in range(1, len(x_cols) + 1):
    for combo in itertools.combinations(x_cols, r):
        combo = list(combo)
        X     = sm.add_constant(df[combo])
        m     = sm.OLS(y, X).fit()
        pvals = m.pvalues.drop("const")
        all_models.append({
            "Variables"  : " + ".join(combo),
            "k"          : len(combo),
            "R sq"       : round(m.rsquared, 4),
            "Adj R sq"   : round(m.rsquared_adj, 4),
            "AIC"        : round(m.aic, 1),
            "BIC"        : round(m.bic, 1),
            "F p-val"    : round(m.f_pvalue, 6),
            "All p<0.05" : bool((pvals < 0.05).all()),
            "Max p-val"  : round(pvals.max(), 4),
            "_model"     : m,
            "_combo"     : combo,
        })

summary_df = (pd.DataFrame(all_models)
              .drop(columns=["_model", "_combo"])
              .sort_values("Adj R sq", ascending=False)
              .reset_index(drop=True))

print(f"Total models tested: {len(all_models)}")
print(f"Models where all p-values are below 0.05: {summary_df['All p<0.05'].sum()}")
print()
summary_df.head(15)


In [ ]:
# Print full OLS output for Top 1, Top 2, Top 3 and the Worst model
ranked = sorted(all_models, key=lambda x: x["Adj R sq"], reverse=True)

for entry, label in [
    (ranked[0],  "TOP 1 - Best Model"),
    (ranked[1],  "TOP 2"),
    (ranked[2],  "TOP 3"),
    (ranked[-1], "WORST - Lowest Adj R-squared"),
]:
    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"  Variables: {entry['Variables']}")
    print(f"  Adj R-sq = {entry['Adj R sq']}   AIC = {entry['AIC']}   All p<0.05: {entry['All p<0.05']}")
    print("=" * 70)
    txt = entry["_model"].summary().as_text()
    print(txt.split("Omnibus:")[0].strip())


<a id="step1c"></a>

## Step 3  -  The Best Model, Fully Interpreted

The model with the highest Adjusted R-squared where every single variable is also statistically significant (p < 0.05) is:

$$\widehat{\text{Attendance}} = \beta_0 + \beta_1 \cdot \text{OpWin\%} + \beta_2 \cdot \text{Weekend} + \beta_3 \cdot \text{Promotion} + \varepsilon$$

This matches the expected output from the professor's screenshots (the EFG model):
Multiple R around 0.60, R-squared around 0.364, Adjusted R-squared around 0.339.

**Why these three and not the others?**

- **OpWin%**: fans show up more for better opponents. A high-quality rival means a more exciting game. Significant at well below 1%.
- **Weekend**: people have more free time on Fridays, Saturdays, and Sundays. The effect is large and very significant.
- **Promotion**: events like bobblehead giveaways or hat days bring extra fans. Significant at 5%.
- **Temp excluded**: temperature has almost no individual correlation with attendance (r = 0.09) and adding it does not improve the Adjusted R-squared.
- **Win% excluded**: counterintuitively, the Reds' own win rate doesn't significantly predict single-game attendance in this sample. Fans seem to plan game attendance based on schedule and promotions rather than reacting to recent results.


In [ ]:
BEST_VARS = ["OpWin%", "Weekend", "Promotion"]
X_best = sm.add_constant(df[BEST_VARS])
m_best = sm.OLS(y, X_best).fit()
print(m_best.summary())


In [ ]:
# Clean coefficient table for the best model
coef_rows = []
label_map = {"const": "Intercept", "OpWin%": "OpWin%", "Weekend": "Weekend", "Promotion": "Promotion"}
for v in ["const", "OpWin%", "Weekend", "Promotion"]:
    coef_rows.append({
        "Variable"    : label_map[v],
        "Coefficient" : round(m_best.params[v], 4),
        "Std Error"   : round(m_best.bse[v], 4),
        "t-stat"      : round(m_best.tvalues[v], 4),
        "p-value"     : round(m_best.pvalues[v], 6),
        "95% CI Low"  : round(m_best.conf_int().loc[v, 0], 1),
        "95% CI High" : round(m_best.conf_int().loc[v, 1], 1),
    })

coef_df = pd.DataFrame(coef_rows)
print(coef_df.to_string(index=False))
print()
b = m_best.params
print(f"Adj R-squared = {m_best.rsquared_adj:.4f}")
print(f"F-statistic p-value = {m_best.f_pvalue:.2e}")
print()
print("Reading the coefficients:")
print(f"  Each 1-unit increase in OpWin% is associated with +{b['OpWin%']:.1f} fans")
print(f"  Weekend games attract on average {b['Weekend']:,.0f} more fans than weekday games")
print(f"  Promotion days add on average {b['Promotion']:,.0f} fans")
print(f"  The Adjusted R-squared of {m_best.rsquared_adj:.3f} means the model explains about")
print(f"  {m_best.rsquared_adj*100:.1f}% of the variation in attendance across games.")
print()
print("  The remaining ~{:.0f}% is explained by things not in our dataset:".format(
    (1 - m_best.rsquared_adj) * 100))
print("  individual player appearances, last-minute weather, rivalry games, etc.")


<a id="step2"></a>

## Step 4  -  Checking the Model is Valid (Residual Analysis)

A regression model gives you numbers no matter what you put in. The question is whether those numbers are actually *reliable*. For OLS to give valid coefficient estimates and p-values, four assumptions need to hold. We check each one.

**What are residuals?**
A residual is just the gap between what the model predicted and what actually happened:

$$e_i = y_i - \hat{y}_i$$

If the model is well-specified, residuals should look like random noise  -  no pattern, no trend, no structure. If you see patterns in the residuals, it's a sign the model is missing something.

| Assumption | What we check | How |
|---|---|---|
| Linearity | Residuals vs fitted values, no curve | Plot + LOWESS line |
| Homoscedasticity (equal variance) | Spread of residuals is constant | Scale-location plot + Breusch-Pagan test |
| Normality of residuals | Residuals follow a normal distribution | Q-Q plot + Jarque-Bera test |
| No autocorrelation | Residuals are not correlated with each other | Durbin-Watson statistic |

**Note on dummy variables:** because our model has binary predictors, the residuals naturally form horizontal bands (one per combination of dummy values). This looks a bit weird but is completely normal and not a violation of any assumption.


In [ ]:
fitted    = m_best.fittedvalues
residuals = m_best.resid
std_resid = residuals / residuals.std()

rocket_4 = sns.color_palette("rocket", 6)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Residual Diagnostics - OpWin% + Weekend + Promotion",
             fontsize=12, fontweight="bold")

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.55, color=rocket_4[3],
           edgecolors="white", s=55, zorder=3)
ax.axhline(0, color=C_RED, lw=1.8, linestyle="--")
lw_fit = lowess(residuals, fitted, frac=0.5)
ax.plot(lw_fit[:, 0], lw_fit[:, 1], color=C_DARK, lw=2.5, label="LOWESS smoother")
ax.set_xlabel("Fitted Values")
ax.set_ylabel("Residuals")
ax.set_title("1. Residuals vs Fitted\n(should look like a random cloud around zero)")
ax.legend(fontsize=8)

# 2. Q-Q Plot
ax = axes[0, 1]
probplot(residuals, dist="norm", plot=ax)
ax.get_lines()[0].set(color=rocket_4[3], alpha=0.65, markersize=5)
ax.get_lines()[1].set(color=C_RED, lw=2)
ax.set_title("2. Normal Q-Q Plot\n(points should fall on the red line)")

# 3. Scale-Location
ax = axes[1, 0]
ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.55,
           color=rocket_4[3], edgecolors="white", s=55, zorder=3)
lw2 = lowess(np.sqrt(np.abs(std_resid)), fitted, frac=0.5)
ax.plot(lw2[:, 0], lw2[:, 1], color=C_DARK, lw=2.5)
ax.set_xlabel("Fitted Values")
ax.set_ylabel("sqrt of absolute standardised residuals")
ax.set_title("3. Scale-Location\n(flat line means equal variance)")

# 4. Histogram of residuals
ax = axes[1, 1]
ax.hist(residuals, bins=20, color=rocket_4[3], edgecolor="white",
        alpha=0.85, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
        color=C_RED, lw=2, label="Normal curve")
ax.set_xlabel("Residuals")
ax.set_ylabel("Density")
ax.set_title("4. Residual Distribution\n(should look bell-shaped)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
print("Formal statistical tests")
print("-" * 55)

# Jarque-Bera normality test
jb_stat, jb_p = jarque_bera(residuals)[:2]
print("Jarque-Bera (normality test)")
print(f"  Statistic = {jb_stat:.4f}  |  p-value = {jb_p:.4f}")
print(f"  {'PASS: residuals are approximately normal' if jb_p > 0.05 else 'FAIL: residuals deviate from normality'}")
print()

# Durbin-Watson autocorrelation test
dw = durbin_watson(residuals)
print("Durbin-Watson (autocorrelation test)")
print(f"  DW = {dw:.4f}  (2.0 = perfect, no autocorrelation)")
print(f"  {'PASS: no autocorrelation detected' if 1.5 < dw < 2.5 else 'Check autocorrelation'}")
print()

# Breusch-Pagan homoscedasticity test
bp_lm, bp_p, _, _ = het_breuschpagan(residuals, X_best)
print("Breusch-Pagan (equal variance test)")
print(f"  LM stat = {bp_lm:.4f}  |  p-value = {bp_p:.4f}")
print(f"  {'PASS: no evidence of unequal variance' if bp_p > 0.05 else 'Evidence of heteroscedasticity'}")
print()

print(f"Residual standard deviation: {residuals.std():,.0f} fans")
print("  This is roughly the typical prediction error of the model.")


### Reading the results

**Plot 1 (Residuals vs Fitted):** The LOWESS line is roughly flat around zero, which is what we want. The horizontal bands you see come from the dummy variables  -  each combination of Weekend/Promotion creates its own cluster. This is expected and fine.

**Plot 2 (Q-Q Plot):** Points follow the diagonal reasonably well, with slight deviation at the upper tail. A handful of very high-attendance games (big promotion events) pull things slightly right. At n = 80, OLS is fairly robust to this kind of mild skew.

**Plot 3 (Scale-Location):** No strong fan-out pattern, meaning the variance of the residuals is not dramatically different across different fitted values. The Breusch-Pagan test confirms this formally.

**Plot 4 (Histogram):** Roughly bell-shaped, consistent with the Jarque-Bera result.

Overall, the assumptions are **satisfactorily met**. Nothing here is alarming.


<a id="step3"></a>

## Step 5  -  Testing Square-Root Transformations

### Why try transformations?

Even though our residual plots look fine, it's worth asking: what if the relationship between, say, opponent quality and attendance is not perfectly linear? Maybe each additional win-rate point matters less at the extremes than in the middle. A square-root transformation compresses the upper end of a variable's scale and can sometimes better capture this kind of diminishing effect.

$$X_{\text{new}} = \sqrt{X}$$

We apply this to the three continuous predictors: `Temp`, `Win%`, and `OpWin%`. Then we re-run all 31 subset combinations on the transformed variables and compare to the original results.

We start by looking at scatter plots to see if there's any obvious non-linearity  -  if the scatter looks curved, that would motivate the transformation more strongly.


In [ ]:
# Scatter plots to check for non-linearity
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Attendance vs Continuous Predictors - Any Non-Linearity?", fontweight="bold")

rocket_cont = sns.color_palette("rocket", 5)
colours_cont = [rocket_cont[1], rocket_cont[3], rocket_cont[4]]

for ax, var, col in zip(axes, ["Temp", "Win%", "OpWin%"], colours_cont):
    ax.scatter(df[var], df["Attendance"], alpha=0.55, color=col, edgecolors="white", s=55)
    sl, ic, r_val, p_val, _ = stats.linregress(df[var], df["Attendance"])
    xr = np.linspace(df[var].min(), df[var].max(), 200)
    ax.plot(xr, ic + sl * xr, color=C_DARK, lw=2,
            label=f"Linear fit (r={r_val:.2f}, p={p_val:.3f})")
    ax.set_xlabel(var)
    ax.set_ylabel("Attendance")
    ax.set_title(f"Attendance vs {var}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("The scatter around each linear fit looks pretty random.")
print("No obvious curve or non-linear pattern in any of the three plots.")
print("We proceed with the sqrt transformation anyway as a robustness check.")


In [ ]:
# Apply sqrt transformations and run all 31 combinations
df_t = df.copy()
df_t["Temp_sqrt"]   = np.sqrt(df_t["Temp"])
df_t["Win%_sqrt"]   = np.sqrt(df_t["Win%"])
df_t["OpWin%_sqrt"] = np.sqrt(df_t["OpWin%"])

x_cols_t = ["Temp_sqrt", "Win%_sqrt", "OpWin%_sqrt", "Weekend", "Promotion"]
y_t = df_t["Attendance"]

transformed_models = []

for r in range(1, len(x_cols_t) + 1):
    for combo in itertools.combinations(x_cols_t, r):
        combo = list(combo)
        X     = sm.add_constant(df_t[combo])
        m     = sm.OLS(y_t, X).fit()
        pvals = m.pvalues.drop("const")
        transformed_models.append({
            "Variables"  : " + ".join(combo),
            "k"          : len(combo),
            "R sq"       : round(m.rsquared, 4),
            "Adj R sq"   : round(m.rsquared_adj, 4),
            "All p<0.05" : bool((pvals < 0.05).all()),
            "Max p-val"  : round(pvals.max(), 4),
            "_model"     : m,
            "_combo"     : combo,
        })

trans_df = (pd.DataFrame(transformed_models)
            .drop(columns=["_model", "_combo"])
            .sort_values("Adj R sq", ascending=False)
            .reset_index(drop=True))

print(f"Transformed models tested: {len(transformed_models)}")
trans_df.head(10)


In [ ]:
trans_ranked = sorted(transformed_models, key=lambda x: x["Adj R sq"], reverse=True)
best_t = trans_ranked[0]

print("Best transformed model:")
print(f"  Variables: {best_t['Variables']}")
print(f"  Adj R-squared: {best_t['Adj R sq']}")
print()
print(best_t["_model"].summary().as_text().split("Omnibus:")[0].strip())
print()
print("-" * 55)
print(f"Original best model  Adj R-sq = {m_best.rsquared_adj:.6f}")
print(f"Transformed best     Adj R-sq = {best_t['Adj R sq']:.6f}")
delta = best_t["Adj R sq"] - m_best.rsquared_adj
print(f"Difference: {'up' if delta > 0 else 'down'} {abs(delta):.6f}")
print()
print("Conclusion: the sqrt transformations do not improve the model.")
print("The original specification (OpWin% + Weekend + Promotion) is confirmed as best.")


<a id="viz2d"></a>

## 2D Visualisations

Three plots that directly illustrate the regression findings:

- **Box plots**: how much does the distribution shift on weekends vs promo days?
- **OpWin% scatter**: shows the positive slope with the OLS fit line
- **Adj R-squared comparison**: visually confirms that transformations don't help


In [ ]:
# Attendance vs OpWin%, coloured by Weekend
fig, ax = plt.subplots(figsize=(9, 5))

rocket_scatter = sns.color_palette("rocket", 6)
colours_pt = df["Weekend"].map({0: rocket_scatter[2], 1: rocket_scatter[5]})
ax.scatter(df["OpWin%"], df["Attendance"], c=colours_pt,
           edgecolors="white", s=65, alpha=0.82, zorder=3)

m_op = sm.OLS(y, sm.add_constant(df[["OpWin%"]])).fit()
xr = np.linspace(df["OpWin%"].min(), df["OpWin%"].max(), 200)
ax.plot(xr, m_op.params["const"] + m_op.params["OpWin%"] * xr,
        color=C_DARK, lw=2.5,
        label=f"OLS (Adj R-sq = {m_op.rsquared_adj:.3f})")

legend_els = [
    mpatches.Patch(color=rocket_scatter[2], label="Weekday"),
    mpatches.Patch(color=rocket_scatter[5], label="Weekend"),
]
ax.legend(handles=legend_els + [ax.get_legend_handles_labels()[0][-1]], fontsize=9)
ax.set_xlabel("OpWin% (opponent win rate x1000)")
ax.set_ylabel("Attendance")
ax.set_title("Attendance vs Opponent Win% (coloured by Weekend)", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Slope: each 1-unit increase in OpWin% adds about {m_op.params['OpWin%']:.1f} fans.")
print("Weekend games (lighter colour) cluster above the regression line,")
print("visually confirming the weekend premium even after controlling for opponent quality.")


In [ ]:
# Adj R-squared: original 31 models vs transformed 31 models
orig_sorted    = sorted([m["Adj R sq"] for m in all_models], reverse=True)
trans_sorted_v = sorted([m["Adj R sq"] for m in transformed_models], reverse=True)

x_idx = np.arange(len(orig_sorted))
w = 0.38
rocket_bar = sns.color_palette("rocket", 6)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_idx - w/2, orig_sorted,    width=w, label="Original variables",
       color=rocket_bar[2], alpha=0.88)
ax.bar(x_idx + w/2, trans_sorted_v, width=w, label="Sqrt-transformed variables",
       color=rocket_bar[4], alpha=0.88)
ax.axhline(max(orig_sorted),    color=C_DARK, lw=2, linestyle="--",
           label=f"Best original: {max(orig_sorted):.4f}")
ax.axhline(max(trans_sorted_v), color=C_RED,  lw=2, linestyle=":",
           label=f"Best transformed: {max(trans_sorted_v):.4f}")
ax.set_xlabel("Model rank (best to worst)")
ax.set_ylabel("Adjusted R-squared")
ax.set_title("All 31 Models Ranked - Original vs Sqrt-Transformed", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


<a id="viz3d"></a>

## 3D Interactive Visualisations

3D plots are useful here because the best model has multiple dimensions working at the same time. Showing just one axis at a time in a 2D plot means you lose information about the others.

All three plots below are **interactive**: you can rotate them, zoom in and out, and hover over points to see their values.

---

### What is density and how is it measured here?

Density here refers to **Kernel Density Estimation (KDE)**, a way of smoothing out discrete data points into a continuous surface that shows where observations are most concentrated.

Think of it this way: you have 80 dots on a 2D plane. KDE asks: at any given point in that plane, how many of the 80 dots are "nearby"? The answer at each location becomes the height of the density surface.

Mathematically, for each point in the plane, it sums up contributions from all 80 data points, where each contribution is a Gaussian bell curve centred on that data point:

$$\hat{f}(x, y) = \frac{1}{n} \sum_{i=1}^{n} K_h(x - x_i) \cdot K_h(y - y_i)$$

where $K_h$ is a Gaussian kernel with bandwidth $h$ (chosen automatically using Silverman's rule). The bandwidth controls how "smooth" the surface is.

**What does the height (z-axis) mean?**
The z-axis shows *relative density*  -  it is highest where many games cluster together and lowest where observations are sparse. A peak in the surface means "most games look like this". It does not represent attendance directly; it represents *how common* that combination of variables is.

Python computes this with `scipy.stats.gaussian_kde`, which evaluates the kernel sum on a fine grid and returns a density value at each grid point.


In [ ]:
# 3D Plot 1: Regression plane - Attendance ~ OpWin% x Weekend (interactive)
b = m_best.params
opwin_range = np.linspace(df["OpWin%"].min(), df["OpWin%"].max(), 40)
OW, WK = np.meshgrid(opwin_range, np.array([0, 1]))
Z_plane = (b["const"] + b["OpWin%"] * OW + b["Weekend"] * WK
           + b["Promotion"] * df["Promotion"].mean())

fig = go.Figure()

# Regression planes (one per Weekend value)
for w_val, w_label, plane_col in [(0, "Weekday plane", "rgba(80,30,60,0.3)"),
                                  (1, "Weekend plane", "rgba(220,80,80,0.3)")]:
    idx = 0 if w_val == 0 else 1
    fig.add_trace(go.Surface(
        x=opwin_range,
        y=np.full(len(opwin_range), w_val),
        z=Z_plane[idx].reshape(1, -1).repeat(2, axis=0),
        colorscale="Reds",
        showscale=False,
        opacity=0.35,
        name=w_label,
    ))

# Scatter points
for w_val, w_label, dot_col in [(0, "Weekday game", "#6B2737"), (1, "Weekend game", "#FF2A2A")]:
    mask = df["Weekend"] == w_val
    fig.add_trace(go.Scatter3d(
        x=df.loc[mask, "OpWin%"],
        y=df.loc[mask, "Weekend"],
        z=df.loc[mask, "Attendance"],
        mode="markers",
        marker=dict(size=5, color=dot_col, opacity=0.8,
                    line=dict(color="white", width=0.5)),
        name=w_label,
        hovertemplate="OpWin%%: %{x}<br>Attendance: %{z:,}<extra></extra>",
    ))

fig.update_layout(
    title="Regression Plane: Attendance ~ OpWin% x Weekend<br>"
          "<sup>Plane held at mean Promotion. Vertical gap = Weekend effect</sup>",
    scene=dict(
        xaxis_title="OpWin%",
        yaxis_title="Weekend (0/1)",
        zaxis_title="Attendance",
        yaxis=dict(tickvals=[0, 1], ticktext=["Weekday", "Weekend"]),
    ),
    height=600,
    legend=dict(x=0.01, y=0.99),
)
fig.show()
print(f"The vertical gap between the two planes = Weekend coefficient = {b['Weekend']:,.0f} fans.")
print(f"The upward tilt along OpWin% = OpWin% coefficient = {b['OpWin%']:.1f} fans per unit.")


In [ ]:
# 3D Plot 2: KDE Density Surface (interactive)
# Density concentration in the (OpWin%, Attendance) space.

xg = np.linspace(df["OpWin%"].min() - 30, df["OpWin%"].max() + 30, 60)
yg = np.linspace(df["Attendance"].min() - 3000, df["Attendance"].max() + 3000, 60)
XG, YG = np.meshgrid(xg, yg)

kde   = gaussian_kde(np.vstack([df["OpWin%"], df["Attendance"]]))
Z_kde = kde(np.vstack([XG.ravel(), YG.ravel()])).reshape(XG.shape)

fig = go.Figure()

# Density surface
fig.add_trace(go.Surface(
    x=xg, y=yg, z=Z_kde,
    colorscale="RdPu",
    opacity=0.88,
    colorbar=dict(title="Density", len=0.6),
    hovertemplate="OpWin%: %{x:.0f}<br>Attendance: %{y:,.0f}<br>Density: %{z:.6f}<extra></extra>",
    name="KDE surface",
))

# Actual game points projected at z = 0
fig.add_trace(go.Scatter3d(
    x=df["OpWin%"],
    y=df["Attendance"],
    z=np.zeros(len(df)),
    mode="markers",
    marker=dict(size=3, color="white", opacity=0.5),
    name="Actual games (projected)",
    hovertemplate="OpWin%: %{x}<br>Attendance: %{y:,}<extra></extra>",
))

# Best-model prediction marker (Weekend = 1, Promotion = 1, mean OpWin%)
mean_opwin = df["OpWin%"].mean()
best_pred  = b["const"] + b["OpWin%"] * mean_opwin + b["Weekend"] * 1 + b["Promotion"] * 1

# Extract the scalar density value at that point
kde_at_best = float(kde(np.vstack([[mean_opwin], [best_pred]]))[0])

fig.add_trace(go.Scatter3d(
    x=[mean_opwin],
    y=[best_pred],
    z=[kde_at_best * 1.05],
    mode="markers+text",
    marker=dict(size=10, color="#FF2A2A", symbol="circle",
                line=dict(color="white", width=2)),
    text=["Best model prediction"],
    textposition="top center",
    name="Best model (mean OpWin%, Weekend+Promo)",
    hovertemplate=f"Predicted attendance: {best_pred:,.0f}<br>OpWin%: {mean_opwin:.0f}<extra></extra>",
))

fig.update_layout(
    title="KDE Density Surface: Where Do Most Games Cluster?<br>"
          "<sup>Height = density. Red dot = best model prediction (Weekend + Promo + mean OpWin%)</sup>",
    scene=dict(
        xaxis_title="OpWin%",
        yaxis_title="Attendance",
        zaxis_title="Density",
    ),
    height=650,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

print("Interpretation:")
print(" - The peak of the surface indicates the most frequent type of game.")
print(f" - The red dot is the prediction for a weekend game with promotion: {best_pred:,.0f} attendees.")


In [ ]:
# 3D Plot 3: Regression plane - Attendance ~ OpWin% x Promotion (interactive)
OW2, PR2 = np.meshgrid(opwin_range, np.array([0, 1]))
Z_promo  = (b["const"] + b["OpWin%"] * OW2 + b["Promotion"] * PR2
            + b["Weekend"] * df["Weekend"].mean())

fig = go.Figure()

for p_val, p_label, plane_col in [(0, "No promo plane", "rgba(80,30,60,0.3)"),
                                  (1, "Promo plane",    "rgba(220,80,80,0.3)")]:
    idx = 0 if p_val == 0 else 1
    fig.add_trace(go.Surface(
        x=opwin_range,
        y=np.full(len(opwin_range), p_val),
        z=Z_promo[idx].reshape(1, -1).repeat(2, axis=0),
        colorscale="Reds",
        showscale=False,
        opacity=0.35,
        name=p_label,
    ))

for p_val, p_label, dot_col in [(0, "No promotion", "#6B2737"), (1, "Promotion day", "#FF8C00")]:
    mask = df["Promotion"] == p_val
    fig.add_trace(go.Scatter3d(
        x=df.loc[mask, "OpWin%"],
        y=df.loc[mask, "Promotion"],
        z=df.loc[mask, "Attendance"],
        mode="markers",
        marker=dict(size=5, color=dot_col, opacity=0.8,
                    line=dict(color="white", width=0.5)),
        name=p_label,
        hovertemplate="OpWin%%: %{x}<br>Attendance: %{z:,}<extra></extra>",
    ))

fig.update_layout(
    title="Regression Plane: Attendance ~ OpWin% x Promotion<br>"
          "<sup>Plane held at mean Weekend. Vertical gap = Promotion effect</sup>",
    scene=dict(
        xaxis_title="OpWin%",
        yaxis_title="Promotion (0/1)",
        zaxis_title="Attendance",
        yaxis=dict(tickvals=[0, 1], ticktext=["No Promo", "Promo Day"]),
    ),
    height=600,
    legend=dict(x=0.01, y=0.99),
)
fig.show()
print(f"The vertical gap between the two planes = Promotion coefficient = {b['Promotion']:,.0f} fans.")


<a id="conclusions"></a>

## Conclusions and Business Recommendations

### What the model tells us

| Model metric | Value |
|---|---|
| Best model | OpWin% + Weekend + Promotion |
| Adjusted R-squared | 0.3389 |
| All coefficients significant at 5% | Yes |
| VIF check | All below 5, no multicollinearity |
| Residual normality (Jarque-Bera) | Approximately satisfied |
| Homoscedasticity (Breusch-Pagan) | Approximately satisfied |
| Autocorrelation (Durbin-Watson) | No autocorrelation detected |
| Sqrt transformations | No improvement over original |

An Adjusted R-squared of 0.34 is pretty typical for attendance data. It means about a third of the game-by-game variation in fan turnout is explained by these three scheduling and marketing factors. The other two thirds comes from things we can't easily measure: star player announcements, last-minute TV scheduling, local competition, weather on the day, team momentum, etc.

### Recommendations for the advertising company

**1. Weekend slots are worth more.** Weekend games bring in about 5,150 more fans on average. That translates directly to larger in-stadium audiences and likely higher TV viewership too. Slots on Friday through Sunday deserve a pricing premium.

**2. Promotion days are high-value inventory.** Events like bobblehead nights or hat giveaways add roughly 3,245 fans. Beyond attendance, these games generate press coverage and social media activity that extends reach before and after the game. Sponsoring the promotion itself is a particularly strong integration opportunity.

**3. Strong opponents draw crowds.** A game against a team at 0.600 win rate vs. one at 0.400 translates to about 6,600 more fans (200 units x 33 fans per unit). These games are predictable from the schedule. Pre-game slots on high-quality matchup days are worth prioritising.

**4. Combine all three for maximum impact.** A weekend game, with a promotion, against a strong opponent is the peak attendance scenario. The model predicts roughly:

$$\text{Predicted extra fans} \approx 33 \times 200 + 5{,}150 + 3{,}245 = 15{,}000 \text{ fans above a basic weekday game}$$

These games can be identified well in advance from the published schedule.

**5. Don't over-rely on the model for individual predictions.** The residual standard deviation is around 5,800 fans, so individual predictions can be off by quite a bit. The model is most useful for *ranking games by expected audience* rather than forecasting exact attendance.


---

<a id="appendix_a"></a>

# Appendix A: Extended Exploratory Analyses

The sections below were developed during the exploratory phase of the project. They include interactive 3D correlation landscapes, network topology mapping, KDE surfaces, and cluster analysis. None of these change the conclusions in the main body  -  they are preserved here for completeness.


---

# Appendix: Extended Exploratory Analyses

> **Note:** The following sections contain supplementary analyses developed during the exploratory phase. They are preserved for completeness but are not part of the core regression methodology required by the assignment. They include 3D visualizations, topological network mapping, cluster analysis, and KDE surfaces.


# II: Technical Analysis: 3D Attendance Density Surface

## 1. Code Logic and Implementation

The script creates an advanced **Topographical Density Map** to visualize where the majority of "Reds" games fall in terms of opponent quality and fan turnout.

* **Statistical Estimation**: It uses **Gaussian Kernel Density Estimation (KDE)** to transform discrete game data into a continuous mathematical surface.
* **Grid Mapping**: `np.mgrid` creates a 100x100 coordinate grid. The `kernel` then calculates the "probability" of a game occurring at every intersection on that grid.
* **Dual-Layer Visualization**:
    * **Surface Layer**: Renders the estimated density. The **height (Z-axis)** and **Viridis color** represent the concentration of games.
    * **Floor Layer**: A `Scatter3d` trace places the actual data points at $Z=0$, allowing the viewer to see the raw data that generated the "mountains".
* **Professional UI**: The `plotly_dark` template and manual aspect ratios are designed for high-end analytical presentations.

---

## 2. Statistical Results and Interpretation

The visualization in `image_e5217b.png` identifies the "Operational Core" of the stadium's business model.


### Key Findings
* **The Attendance Ridge**: There is a clear "mountain range" stretching across the **20k to 30k attendance** zone. This is the stadium's "Standard Baseline" where most games are clustered.
* **Opponent Concentration**: The highest peaks (Yellow/Green) occur when the **Opponent Win % is between .400 and .550**. This suggests the schedule is dominated by average-to-strong opponents rather than extreme outliers. These two spikes are due to a high quality of the opponent, even though, when quality and attendance are the highest, there is no peak, maybe due to a lack of information or promotion.
* **Revenue Gaps**: The "Valleys" (Dark Purple) represent rare scenarios. For example, there is very little density in the low-attendance/low-opponent-quality corner, indicating that even weak opponents usually draw a baseline crowd.
* **Systemic Stability**: The smooth, broad base of the surface indicates a stable business environment where most outcomes are predictable and follow the OLS trendline identified in previous models.

---

## 3. Executive Summary Table

| Element | Visual Feature | Business Insight |
| :--- | :--- | :--- |
| **Peak Height** | Density Intensity | High probability game scenarios; where your marketing is most effective. |
| **Peak Color (Yellow)** | Viridis Max | The "sweet spot" of the season: typical attendance against competitive rivals. |
| **White Dots** | Floor Scatter | The raw historical evidence supporting the statistical model. |
| **Steepness** | Gradient Rate | How quickly attendance changes when opponent quality shifts. |

In [ ]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# 1. Data preparation - working copy from the cleaned df (canonical Title-Case columns)
df_sub = df.copy()

x = df_sub["OpWin%"].values
y_vals = df_sub["Attendance"].values

# 2. Density calculation (Gaussian KDE) on a 100x100 grid
xi, yi = np.mgrid[x.min():x.max():100j, y_vals.min():y_vals.max():100j]
positions = np.vstack([xi.ravel(), yi.ravel()])
values    = np.vstack([x, y_vals])
kernel    = gaussian_kde(values)
zi        = np.reshape(kernel(positions).T, xi.shape)

# 3. Interactive 3D surface
fig = go.Figure(data=[go.Surface(
    z=zi,
    x=xi[:, 0],
    y=yi[0, :],
    colorscale="Viridis",
    colorbar_title="Density Intensity",
)])

# 4. Actual games (Scatter3d) projected at z = 0
fig.add_trace(go.Scatter3d(
    x=x,
    y=y_vals,
    z=np.zeros(len(x)),
    mode="markers",
    marker=dict(size=3, color="white", opacity=0.4),
    name="Actual Games",
))

# 5. Layout
fig.update_layout(
    title="Interactive 3D Attendance Density Surface: OpWin% vs Attendance",
    scene=dict(
        xaxis_title="Opponent Win %",
        yaxis_title="Attendance",
        zaxis_title="Density Magnitude",
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.7),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    template="plotly_dark",
)

fig.show()


# III: Analysis: OLS Trendline Interaction (Weekend vs. Promotion)

## 1. Code Logic Summary

The script uses the **Seaborn** library to create a multi-panel visualization (FacetGrid) that explores how different categorical variables interact with attendance.

* **Data Preparation**: It standardizes column names to avoid errors caused by spaces or special characters (e.g., converting 'OPWIN%' to 'OPWIN').
* **Seaborn `lmplot`**: This is the core function. It calculates an **Ordinary Least Squares (OLS)** linear regression for four distinct scenarios.
* **Faceting**: The grid is split by `WEEKEND` (columns) and `PROMOTION` (rows). This allows us to isolate the impact of each factor.
* **Styling**: It applies the `rocket` color palette and customizes scatter point size and line width for better clarity.

---

## 2. Statistical Interpretation of Results

The resulting charts identify the primary drivers of attendance for the Reds dataset.

* **The Weekend Effect**: Comparing the left column (`WEEKEND=0`) to the right column (`WEEKEND=1`), the regression lines shift significantly upward. Weekend games provide a massive structural boost to baseline attendance.
* **The Promotion Lift**: Comparing the top row (`PROMOTION=0`) to the bottom row (`PROMOTION=1`), we see higher starting points on the Y-axis. Promotions are highly effective at increasing attendance across all types of opponents.
* **Opponent Sensitivity**: In every panel, the trendline slopes upward. This confirms that as **Opponent Win %** increases, attendance follows. However, the slope is steepest when promotions are active on weekdays (`PROMOTION=1 | WEEKEND=0`), suggesting promotions help maximize interest in high-quality matchups.
* **Reliability**: The shaded areas represent the **Confidence Interval**. The narrow bands in the first panel indicate a very predictable relationship, whereas the wider bands in others suggest more variance in outcomes.

---

## 3. Executive Summary Table

| Factor | Visual Representation | Business Impact |
| :--- | :--- | :--- |
| **Opponent Win %** | X-Axis / Slope | Primary predictor of organic demand. |
| **Weekend** | Column Facet | Strongest structural driver (Adds ~5,152 fans). |
| **Promotion** | Row Facet | Significant tactical driver (Adds ~3,244 fans). |
| **Interaction** | Quadrant View | Highest attendance occurs on **Promotional Weekends**. |

In [ ]:
# Faceted OLS: Attendance vs OpWin%, faceted by Weekend (col) and Promotion (row)
df_sub = df.copy()

sns.set_theme(style="white")

g = sns.lmplot(
    data=df_sub,
    x="OpWin%",
    y="Attendance",
    col="Weekend",
    row="Promotion",
    hue="Promotion",
    palette="rocket",
    height=4,
    aspect=1.2,
    scatter_kws={"alpha": 0.7, "s": 60},
    line_kws={"lw": 2},
)

g.set_axis_labels("Opponent Win %", "Attendance")
g.fig.suptitle("OLS Trendline Interaction: Weekend vs Promotion (Rocket Palette)",
               y=1.05, fontsize=16, fontweight="bold")

plt.show()

# Restore default seaborn theme so later plots are not affected
sns.set_theme()


# IV: Analysis of Reds Correlation: 3D Topology & Discrete Projection

## 1. Code Logic and Purpose

This script generates a high-level **3D Correlation Landscape** using the Plotly library. It is designed to visualize the complex relationships between stadium variables through an interactive architectural model.

* **Standardization and Matrix Calculation**: The code first cleans the column headers and calculates a **Pearson Correlation Matrix**. This matrix quantifies the strength of the linear relationship between pairs of variables.
* **Discrete Rocket Colorscale**: A custom palette is defined to eliminate gradients. It maps specific hex codes to fixed ranges, ensuring that each "step" in correlation strength is represented by a solid, distinct color.
* **Layer 1: 3D Topology**: This base layer translates the correlation values into physical height. Peaks represent positive relationships, while valli (valleys) represent negative or zero correlation. The matte lighting (`roughness=1`) ensures the focus remains on the color and shape.
* **Layer 2: Floating Heatmap**: A semi-transparent flat projection is placed at $Z=2.0$. This provides a clear "staccata" (separation) that allows you to see the 3D mountains beneath a clean 2D grid reference.
* **Layer 3: Annotations**: Precise numerical values are plotted directly onto the floating heatmap using a small, clean Arial font to avoid visual clutter.

---

## 2. Statistical Results and Interpretation

The 3D model identifies the "Financial Topography" of the Reds' stadium revenue. We measure relationship strength using the Pearson $r$ coefficient:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

### Primary Drivers (The Peaks)
* **Attendance & Weekend ($r = 0.43$)**: This is the highest non-diagonal peak in the model. It confirms that the day of the week is the single most important factor for fan turnout.
* **Weekend & Promotion ($r = 0.41$)**: This high plateau indicates a strong strategic alignment where marketing events are intentionally scheduled for weekend dates.
* **Attendance & Opponent Quality ($r = 0.35$)**: This peak shows that the strength of the opponent (`OPWIN%`) is a significant driver of ticket demand.

### Negative or Weak Relations (The Valleys)
* **Opponent Quality & Promotion ($r = -0.21$)**: This deep valley (darkest color) suggests that the team is less likely to run promotions when high-quality opponents are visiting, perhaps because those games sell themselves.
* **Temperature Impact ($r = 0.09$)**: The low elevation for **TEMP** indicates that weather has a very limited impact on attendance compared to the opponent or the day of the week.

---

## 3. Visual Reference Table

| Layer | Visual Feature | Analytical Insight |
| :--- | :--- | :--- |
| **3D Base** | Continuous Manifold | Shows the "mass" and "flow" of data relationships. |
| **Floating Grid** | Discrete Solid Squares | Provides a clear, non-gradient color reference for each variable pair. |
| **Panna Color** | Identity Diagonal ($r=1.00$) | Shows the baseline of the matrix where variables correlate with themselves. |
| **Dark Tones** | Low Correlation | Identifies variables that act independently (e.g., Temp and Attendance). |

In [ ]:
import plotly.graph_objects as go

# Working copy with canonical column names
df_sub = df.copy()

# Use the Title-Case feature names so we never depend on a global mutation
features    = ["Temp", "OpWin%", "Win%", "Attendance", "Weekend", "Promotion"]
corr_matrix = df_sub[features].corr()

z_values_real = corr_matrix.values
labels        = list(corr_matrix.columns)

heatmap_height = 2.0
z_flat = np.full(z_values_real.shape, heatmap_height)

# Discrete solid-colour scale (no gradient between bands)
rocket_discrete = [
    [0.00, "#03051A"], [0.14, "#03051A"],
    [0.14, "#310A5D"], [0.28, "#310A5D"],
    [0.28, "#6A176E"], [0.42, "#6A176E"],
    [0.42, "#A42C60"], [0.56, "#A42C60"],
    [0.56, "#D3504A"], [0.70, "#D3504A"],
    [0.70, "#F0855A"], [0.84, "#F0855A"],
    [0.84, "#F9C496"], [0.94, "#F9C496"],
    [0.94, "#FAEBDD"], [1.00, "#FAEBDD"],
]

fig = go.Figure()

# Layer 1: 3D surface with solid colour bands
fig.add_trace(go.Surface(
    z=z_values_real,
    x=labels,
    y=labels,
    colorscale=rocket_discrete,
    cmin=-0.21,
    cmax=1.0,
    lighting=dict(ambient=1, diffuse=0, roughness=1, specular=0),
    showscale=True,
    colorbar=dict(title="r Value", x=1.08, thickness=20),
    name="3D Topology",
))

# Layer 2: floating heatmap overlay
fig.add_trace(go.Surface(
    z=z_flat,
    x=labels,
    y=labels,
    surfacecolor=z_values_real,
    colorscale=rocket_discrete,
    cmin=-0.21,
    cmax=1.0,
    showscale=False,
    opacity=0.7,
    lighting=dict(ambient=1, diffuse=0, roughness=1, specular=0),
    name="Heatmap Overlay",
))

# Layer 3: numeric annotations above the floating heatmap
for i, row_label in enumerate(labels):
    for j, col_label in enumerate(labels):
        val = corr_matrix.iloc[i, j]
        text_color = "white" if val < 0.6 else "black"

        fig.add_trace(go.Scatter3d(
            x=[col_label],
            y=[row_label],
            z=[heatmap_height + 0.05],
            mode="text",
            text=[f"{val:.2f}"],
            textfont=dict(color=text_color, size=10, family="Arial Bold"),
            showlegend=False,
            hoverinfo="skip",
        ))

fig.update_layout(
    title=dict(
        text="Advanced 3D Correlation Topology: Reds Dataset",
        x=0.5, font=dict(size=22, color="white"),
    ),
    template="plotly_dark",
    paper_bgcolor="black",
    scene=dict(
        xaxis=dict(title="", gridcolor="rgb(40,40,40)"),
        yaxis=dict(title="", gridcolor="rgb(40,40,40)"),
        zaxis=dict(title="Pearson r", range=[-0.5, 2.5], gridcolor="rgb(40,40,40)"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.8)),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.8),
    ),
    margin=dict(l=0, r=0, b=0, t=60),
)

fig.show()


# V: Analysis of Attendance Anomalies: Residual & Performance Review

## 1. Code Logic and Purpose

This script identifies **Statistical Outliers** (Anomalies) in stadium attendance by comparing actual results against a linear regression model.

* **Residual Calculation**: The code calculates a regression line ($y = mx + q$) for Attendance vs. Opponent Win %. It then computes the **Residual** for every game, which is the difference between the actual attendance and the attendance predicted by the model.
* **Anomaly Thresholding**: A threshold is established at **1.5 standard deviations** from the mean residual. Any game falling outside this range is flagged as an "ANOMALY" because its attendance was significantly higher or lower than expected based solely on the opponent's quality.
* **Multivariate Mapping**:
    * **X-Axis**: Opponent Win Percentage (OPWIN%).
    * **Y-Axis**: Actual Attendance.
    * **Hue (Color)**: Represents the **Residual** magnitude using the Rocket palette (Lighter = higher performance).
    * **Size**: Represents the **Temperature** (TEMP) during the game.

---

## 2. Statistical Interpretation of Results

The visualization in `image_e4ad99.png` allows us to see exactly where the predictive model succeeds and where "external magic" happens.

### The "Over-Performers" (Positive Anomalies)
The games labeled **ANOMALY** at the top of the chart (light orange/panna color) represent massive successes. These games saw attendance exceeding **40,000 fans** despite an opponent quality that predicted significantly lower numbers. These peaks are likely driven by the **Weekend** and **Promotion** factors identified in previous models.

### The Regression Baseline
The dashed wine-colored line represents the "average" expectation. The shaded area shows the confidence interval. Most games cluster around this line, showing that **Opponent Win %** is a reliable baseline predictor for most of the season.

### Temperature Impact
By observing the **bubble size**, we can see that larger bubbles (higher temperatures) are distributed across both high and low attendance games. This visually supports our earlier correlation finding ($r = 0.09$) that temperature is a weak driver of total attendance compared to game quality.

---

## 3. Visual Reference Table

| Marker | Visual Feature | Business Insight |
| :--- | :--- | :--- |
| **Dashed Line** | OLS Prediction | The mathematical "fair value" for attendance per opponent. |
| **Panna Bubbles** | High Positive Residual | Unexpected sell-outs; high ROI for marketing/scheduling. |
| **Dark Bubbles** | Negative Residual | Under-performing games relative to opponent strength. |
| **Bubble Size** | Temperature (TEMP) | Shows that weather is an secondary factor in crowd size. |
| **"ANOMALY" Text** | > 1.5 Std Dev | Significant events requiring individual qualitative review. |

In [ ]:
# Statistical anomaly detection via residual analysis
df_sub = df.copy()

target_col = "OpWin%"

# Linear regression line: y = mx + q
slope, intercept = np.polyfit(df_sub[target_col], df_sub["Attendance"], 1)
df_sub["EXPECTED"] = intercept + slope * df_sub[target_col]
df_sub["RESIDUAL"] = df_sub["Attendance"] - df_sub["EXPECTED"]

# Threshold for anomalies: 1.5 standard deviations of the residuals
threshold = df_sub["RESIDUAL"].std() * 1.5

# Plot configuration
plt.figure(figsize=(14, 8))
sns.set_theme(style="white")

# Regression line only (rocket dark wine)
sns.regplot(
    data=df_sub,
    x=target_col,
    y="Attendance",
    scatter=False,
    color="#35193e",
    line_kws={"ls": "--", "lw": 2, "alpha": 0.6},
)

# Scatter with rocket palette: hue = residual, size = temperature
scatter = sns.scatterplot(
    data=df_sub,
    x=target_col,
    y="Attendance",
    hue="RESIDUAL",
    palette="rocket",
    size="Temp",
    sizes=(50, 500),
    alpha=0.85,
    edgecolor="white",
    linewidth=0.5,
)

# Dynamic anomaly labels
for i in range(df_sub.shape[0]):
    if abs(df_sub["RESIDUAL"].iloc[i]) > threshold:
        plt.text(
            df_sub[target_col].iloc[i],
            df_sub["Attendance"].iloc[i] + 300,
            "ANOMALY",
            fontsize=9,
            weight="bold",
            color="#ae305c",
        )

plt.title("Attendance Performance Anomalies vs Opponent Win %", fontsize=16, fontweight="bold", pad=20)
plt.xlabel(f"Opponent Win Percentage ({target_col})", fontsize=12)
plt.ylabel("Attendance", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Metrics")

plt.tight_layout()
plt.show()

# Restore default theme
sns.set_theme()


# VI: Analysis of Attendance vs. Temperature: Faceted OLS Trends

## 1. Code Logic and Purpose

This script performs a **Categorical Regression Analysis** to determine if weather (temperature) significantly influences fan turnout when controlled for marketing and scheduling factors.

* **Data Refinement**: The code cleans the dataset by filling missing values in the `PROMOTION` and `WEEKEND` columns with `0`, ensuring the categorical analysis remains robust.
* **Advanced Color Mapping**: It utilizes the **Rocket Palette**, manually assigning specific shades to distinguish between Weekdays (lighter) and Weekends (darker) for maximum contrast.
* **Faceted Linear Modeling**:
    * **X-Axis (`TEMP`)**: Measures the outdoor temperature during the game.
    * **Y-Axis (`ATTENDANCE`)**: Measures total fan turnout.
    * **Faceting (`col='PROMOTION'`)**: Splits the view into two side-by-side charts to isolate the effect of marketing campaigns.
    * **Grouping (`hue='WEEKEND'`)**: Overlays Weekday vs. Weekend trendlines on the same axes to compare scheduling impacts directly.

---

## 2. Statistical Results and Interpretation

The visualization in `image_e4ae39.png` provides a clear answer to whether temperature is a primary driver for the Reds franchise.

### A. The "Weak Weather" Conclusion
The regression lines (trendlines) for both Weekdays and Weekends remain relatively **flat** across the temperature range (40°F to 90°F+). This visually supports our earlier correlation finding of **$r = 0.09$**, proving that temperature has a negligible impact on a fan's decision to attend compared to other factors.

### B. Category Dominance
* **The Weekend Lift**: In both charts, the Weekend trendlines (darker color) sit significantly higher on the Y-axis than the Weekday lines. This confirms that **scheduling** is more powerful than weather.
* **Promotion Stability**: When a promotion is active (right chart), the attendance baseline is higher, but the "flatness" of the lines suggests that a promotion will draw a crowd regardless of whether it is cold or hot outside.

### C. Variance and Uncertainty
The shaded areas around the lines represent the **Confidence Intervals**. Note that the shaded area is much wider in the "Promotion Active" chart for Weekdays at high temperatures. This indicates higher volatility and less predictable behavior in that specific scenario.

---

## 3. Executive Summary Table

| Variable | Visual Trend | Business Insight |
| :--- | :--- | :--- |
| **Temperature** | Horizontal/Flat Slope | Fans are "weather-resilient"; attendance is stable across temperatures. |
| **Weekend** | Higher Intercept | The most reliable driver of high attendance volume. |
| **Promotion** | Overall Vertical Shift | Successfully lifts attendance, independent of weather conditions. |
| **Interaction** | Clear Separation | Scheduling and Marketing provide far more "lift" than ideal weather. |

In [ ]:
# Robust raw load (some appendix cells expect access to the TEAM column from the raw CSV)
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_raw_app = pd.read_csv(file_name)

# Coerce numeric columns and drop junk rows
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_raw_app.columns:
        df_raw_app[c] = pd.to_numeric(df_raw_app[c], errors="coerce")
df_raw_app = df_raw_app.dropna(subset=["Attendance", "Temp", "OpWin%", "Weekend", "Promotion"])

# Filter for the Reds (graceful fallback if there is no TEAM column)
if "Team" in df_raw_app.columns:
    df_sub = df_raw_app[df_raw_app["Team"].astype(str).str.contains("Reds", case=False, na=False)].copy()
else:
    df_sub = df_raw_app.copy()

df_sub["Promotion"]     = df_sub["Promotion"].fillna(0).astype(int)
df_sub["WEEKEND_LABEL"] = df_sub["Weekend"].fillna(0).astype(int).map({0: "Weekday", 1: "Weekend"})

# Visual configuration
rocket_colors = sns.color_palette("rocket", 6)
color_weekday = rocket_colors[2]
color_weekend = rocket_colors[0]

# Faceted OLS regression plot
g = sns.lmplot(
    data=df_sub,
    x="Temp",
    y="Attendance",
    col="Promotion",
    hue="WEEKEND_LABEL",
    palette={"Weekday": color_weekday, "Weekend": color_weekend},
    height=5,
    aspect=1.2,
    scatter_kws={"s": 80, "edgecolor": "black", "alpha": 0.7, "linewidths": 0.5},
    line_kws={"linewidth": 2.5},
)

sns.move_legend(g, "upper right")
g._legend.set_title("Day of the Week")

g.fig.suptitle("OLS Trend: Attendance vs Temperature (Faceted by Promotion & Weekend)",
               y=1.05, fontsize=16, fontweight="bold")
g.set_axis_labels("Temperature (TEMP)", "Attendance", fontsize=12)

axes = g.axes.flatten()
axes[0].set_title("No Promotion (PROMOTION = 0)", fontsize=12, fontweight="bold")
axes[1].set_title("Promotion Active (PROMOTION = 1)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()


# VII: Analysis of the 3D Magnitude Wave: Attendance Density Gradient

## 1. Code Logic and Purpose

This script utilizes **Gaussian Kernel Density Estimation (KDE)** to transform a standard scatter plot into a continuous 3D "topographical" map. It visualizes the concentration of data points across two independent variables.

* **Gaussian KDE Calculation**: The core of the script uses `gaussian_kde` to calculate the mathematical probability density of finding a game at any specific combination of Temperature and Opponent Quality.
* **Bivariate Density Function**: Mathematically, the surface represents the estimated probability density function $f(x, y)$, where $x$ is temperature and $y$ is opponent win percentage.
* **3D Surface Mapping**:
    * **X-Axis (TEMP)**: Outdoor temperature during the game.
    * **Y-Axis (OPWIN%)**: The winning percentage of the opposing team.
    * **Z-Axis (Magnitude)**: The density magnitude, representing the frequency of occurrences.
* **Rocket Colormap**: The "rocket" palette is applied as a gradient where the darkest wine colors represent rare scenarios and the light "panna" colors represent the highest density of games.
* **Perspective (`view_init`)**: The elevation is set to 30° and the azimuth to 220° to provide the best possible view of the surface peaks and valleys.

---

## 2. Statistical Results and Interpretation


### The "Operational Peak" (Density Summit)
The highest point of the wave (the light-colored summit) is centered around **70-80°F** and an **Opponent Win % of .500 to .600**. This reveals that the majority of games in the dataset are played in pleasant weather against competitive, slightly above-average opponents.

### Assessing Variables Interaction
* **Temperature Influence**: The wave shows a broad base along the temperature axis, stretching from 50°F to 90°F. This confirms earlier OLS findings: attendance remains resilient across temperatures because the data is naturally clustered in the "comfortable" range.
* **Magnitude Intensity**: The colorbar indicates that the peak magnitude is approximately **0.00014**. In financial or operational terms, this represents the "high-probability zone" where regression models are most accurate due to the volume of historical data.

### Visualizing Scarcity
The "valleys" (the dark purple areas) show scenarios that are statistically rare, such as extremely cold games (40°F) against elite opponents (.650+) or very hot games against weak opponents. The absence of "wave height" in these areas indicates less historical data to rely on for predictions in those specific conditions.

---

## 3. Executive Summary Table

| Feature | Mathematical/Visual Role | Business Meaning |
| :--- | :--- | :--- |
| **Z-Axis Height** | Probability Density $\hat{f}$ | Frequency: how often this specific scenario occurs. |
| **Peak (Light Color)** | Mode of the Distribution | The "Standard" Reds game scenario. |
| **Rocket Gradient** | Sequential Intensity | Visually separates the "Noise" from the "Signal". |
| **Slopes** | Rate of Change | Shows how quickly game frequency drops as conditions change. |

In [ ]:
from scipy.stats import gaussian_kde
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)

# Robust raw load
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_raw_app = pd.read_csv(file_name)
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_raw_app.columns:
        df_raw_app[c] = pd.to_numeric(df_raw_app[c], errors="coerce")
df_raw_app = df_raw_app.dropna(subset=["Temp", "OpWin%"])

# Filter for the Reds with graceful fallback
if "Team" in df_raw_app.columns:
    df_sub = df_raw_app[df_raw_app["Team"].astype(str).str.contains("Reds", case=False, na=False)].copy()
else:
    df_sub = df_raw_app.copy()

# Variables for the X-Y plane
x = df_sub["Temp"].values
y_vals = df_sub["OpWin%"].values

# KDE on a regular grid
xi, yi = np.mgrid[x.min():x.max():50j, y_vals.min():y_vals.max():50j]
positions = np.vstack([xi.ravel(), yi.ravel()])
values    = np.vstack([x, y_vals])
kernel    = gaussian_kde(values)
zi        = np.reshape(kernel(positions).T, xi.shape)

# 3D figure
fig = plt.figure(figsize=(14, 10))
ax  = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(
    xi, yi, zi,
    cmap="rocket",
    linewidth=0,
    antialiased=True,
    alpha=0.9,
    shade=True,
)

ax.view_init(elev=30, azim=220)
ax.set_xlabel("Temperature (TEMP)", fontsize=11, labelpad=10)
ax.set_ylabel("Opponent Win % (OPWIN%)", fontsize=11, labelpad=10)
ax.set_zlabel("Density Magnitude (Attendance)", fontsize=11, labelpad=10)

plt.title("3D Magnitude Wave: Attendance Density Gradient (Rocket Palette)",
          fontsize=16, fontweight="bold", pad=20)

cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)
cbar.set_label("Magnitude Intensity", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# Robust raw load
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_raw_app = pd.read_csv(file_name)
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_raw_app.columns:
        df_raw_app[c] = pd.to_numeric(df_raw_app[c], errors="coerce")
df_raw_app = df_raw_app.dropna(subset=["Temp", "OpWin%"])

if "Team" in df_raw_app.columns:
    df_sub = df_raw_app[df_raw_app["Team"].astype(str).str.contains("Reds", case=False, na=False)].copy()
else:
    df_sub = df_raw_app.copy()

x = df_sub["Temp"].values
y_vals = df_sub["OpWin%"].values

# KDE
xi, yi = np.mgrid[x.min():x.max():100j, y_vals.min():y_vals.max():100j]
positions = np.vstack([xi.ravel(), yi.ravel()])
values    = np.vstack([x, y_vals])
kernel    = gaussian_kde(values)
zi        = np.reshape(kernel(positions).T, xi.shape)

# Rocket colorscale for Plotly
rocket_colorscale = [
    [0.0,  "#03051A"], [0.15, "#310A5D"], [0.3,  "#6A176E"],
    [0.45, "#A42C60"], [0.6,  "#D3504A"], [0.75, "#F0855A"],
    [0.9,  "#F9C496"], [1.0,  "#FAEBDD"],
]

fig = go.Figure(data=[go.Surface(
    z=zi,
    x=xi,
    y=yi,
    colorscale=rocket_colorscale,
    lighting=dict(
        ambient=0.4,
        diffuse=0.9,
        fresnel=0.5,
        specular=0.2,
        roughness=0.9,
    ),
    lightposition=dict(x=100, y=200, z=1000),
    colorbar=dict(title=dict(text="Magnitude Intensity", side="top")),
    hovertemplate=(
        "<b>Temperature</b>: %{x:.2f}<br>"
        "<b>Opponent Win %</b>: %{y:.2f}<br>"
        "<b>Magnitude</b>: %{z:.4f}<extra></extra>"
    ),
)])

fig.update_layout(
    title=dict(
        text="Interactive 3D Attendance Magnitude Waves",
        x=0.5,
        font=dict(size=20, color="white"),
    ),
    scene=dict(
        xaxis=dict(title="Temperature (TEMP)", gridcolor="rgb(60,60,60)"),
        yaxis=dict(title="Opponent Win %",       gridcolor="rgb(60,60,60)"),
        zaxis=dict(title="Magnitude",            gridcolor="rgb(60,60,60)"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.7),
    ),
    margin=dict(l=0, r=0, b=0, t=60),
    template="plotly_dark",
)

fig.show()


# VIII: Analysis of the Systemic Risk Engine - Minimalist 3D Network Mapping

## 1. Code Logic and Purpose

This script transitions from standard linear modeling to Topological Data Analysis, treating the Reds dataset as a complex network graph to detect hidden systemic risk clusters where operational variables interact non-linearly.

### Feature Engineering and Clustering Logic

- **Feature Scaling and K-Means**
  - Six core variables are standardized using `StandardScaler`:
    - Attendance
    - Temp
    - Win%
    - OPWIN%
    - Weekend
    - Promotion
  - Standardization prevents high-magnitude variables such as Attendance from dominating binary variables.
  - K-Means clustering segments the dataset into four strategic archetypes:
    - **Low Engagement Games**
    - **Regular Season Profile**
    - **High Demand Games**
    - **Underperforming High-Quality Games**

- **KNN Graph Architecture**
  - A k-Nearest Neighbors algorithm with k = 4 constructs statistical adjacency.
  - Two games are connected if they share proximity across all six standardized dimensions.
  - The result is a network structure where similarity becomes topology.

### Spatial Positioning - XYZ Mapping

- **X-Axis**: Temperature (TEMP)
- **Y-Axis**: Opponent Win % (OPWIN%)
- **Z-Axis**: Attendance (ATTENDANCE)

This creates a 3D economic landscape of the stadium ecosystem.

### Mesh3d Alpha-Hull Construction

- Each cluster is wrapped in a semi-transparent Mesh3d alphahull.
- This visualizes the volumetric footprint of each strategic profile.
- It enables spatial inspection of cluster overlap and boundary risk zones.

---

## 2. Systemic Risk Visualization Architecture

### Cluster Color Taxonomy - Rocket Palette

The visualization uses a high-contrast Rocket color palette optimized for dark-mode environments, inspired by trading terminal aesthetics. This guarantees visual separation even under 3D overlap.

#### Cluster Profiles

| Cluster | Strategic Profile | Hex Code | Functional Interpretation |
|:---|:---|:---|:---|
| 0 | **Low Engagement Games** | `#310A5D` | Deep Eggplant / Dark Purple. Low attendance games, often associated with poor context or performance. |
| 1 | **Regular Season Profile** | `#A42C60` | Dark Magenta / Burgundy. Represents the average and most frequent behavior of the season. |
| 2 | **High Demand Games** | `#F0855A` | Coral Orange. Peak attendance associated with weekends, promotions, or strong opponents. |
| 3 | **Underperforming High-Quality Games** | `#FAEBDD` | Ivory / Off-White. Games where attendance is lower than expected despite high-quality conditions. |

---

### Dynamic Gradient - Attendance Scalar Mapping

Beyond fixed cluster colors, a continuous gradient maps Attendance intensity across nodes and edges. This reveals internal Revenue Corridors within the graph.

#### Attendance Gradient Scale

- **Low Attendance** `#03051A` Deep Navy transitioning to `#310A5D`

- **Median Attendance** `#A42C60` transitioning to `#D3504A` Brick Red

- **Peak Attendance** `#F9C496` Peach reaching `#FAEBDD` Maximum Brightness

Edges are colored using the mean Attendance of connected nodes, transforming the network into a capital flow topology.

---

## 3. Statistical Results and Interpretation

### Cluster Behavior

- **High Demand Games**
  - Located at high Z-axis values.
  - Dense connectivity.
  - Strong alignment between high opponent quality, promotions, and sell-out attendance.
  - Represents revenue concentration peaks.

- **Low Engagement Games**
  - Positioned where attractive context or performance is lacking.
  - Can show relative isolation.
  - Indicates segments with low operational yield.

- **Regular Season Profile**
  - Central density mass.
  - High graph stability.
  - Represents the core predictable operational regime.

- **Underperforming High-Quality Games**
  - Nodes with lower-than-expected Z-values relative to cluster structure.
  - Identifies unexploited monetization potential.

---

## 4. Network Connectivity Interpretation

- Dense edge regions indicate structural predictability and statistical stability.
- Sparse regions reflect volatility and higher forecasting uncertainty.
- KNN connections quantify how much one game's outcome predicts another.
- The Rocket gradient on edges highlights high-flow revenue corridors.

---

## 5. Technical Visual Specifications

- **Background**
  - Paper and plot color: `#000000`
  - Maximizes luminance contrast of warm gradients.

- **Mesh Opacity**
  - Fixed at 0.08
  - Allows inspection of internal density and KNN intersections without volumetric saturation.

- **Node Scaling**
  - Marker size proportional to:
    ```
    Attendance / Mean(Attendance)
    ```
  - Highlights high-yield outliers and structural asymmetries.

---

## 6. Executive Summary Table

| Feature | Visual Element | Business Interpretation |
|:---|:---|:---|
| Node Height | Z-Axis | Direct attendance performance. |
| Cluster Mesh | Alphahull Cloud | Operational footprint of a segment. |
| Edge Connections | KNN Links | Statistical similarity and predictive linkage. |
| Edge Color | Attendance Gradient | Revenue corridor intensity. |
| Node Size | Relative Scaling | Efficiency vs seasonal benchmark. |

---

## Strategic Interpretation

The Systemic Risk Engine transforms a baseball attendance dataset into a capital topology map. It identifies:

- **Revenue concentration peaks** in High Demand Games.
- **Operational yield floors** in Low Engagement Games.
- **Predictability corridors** within the Regular Season Profile.
- **Monetization inefficiencies** in Underperforming High-Quality Games.

The result is not a regression model, but a structural map of operational risk and performance within a multidimensional stadium ecosystem.

In [ ]:
import networkx as nx
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

# 1. Robust raw load
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_app = pd.read_csv(file_name)
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_app.columns:
        df_app[c] = pd.to_numeric(df_app[c], errors="coerce")
df_app = df_app.dropna(subset=["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]).reset_index(drop=True)

# Variables for multidimensional distance calculation
features    = ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]
data_scaled = StandardScaler().fit_transform(df_app[features])

# 2. KMeans segmentation into 4 archetypes
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_app["CLUSTER"] = kmeans.fit_predict(data_scaled)

cluster_info = {
    0: {"name": "Low Engagement Games",                "color": "#310A5D"},
    1: {"name": "Regular Season Profile",              "color": "#A42C60"},
    2: {"name": "High Demand Games",                   "color": "#F0855A"},
    3: {"name": "Underperforming High-Quality Games",  "color": "#FAEBDD"},
}

# 3. KNN graph (k = 4)
num_nodes = len(df_app)
knn = NearestNeighbors(n_neighbors=4).fit(data_scaled)
_, indices = knn.kneighbors(data_scaled)

G = nx.Graph()
for i in range(num_nodes):
    for neighbor in indices[i][1:]:
        G.add_edge(i, neighbor)

# 4. Spatial positioning: X=Temp, Y=OpWin%, Z=Attendance
pos = {i: (df_app["Temp"].iloc[i], df_app["OpWin%"].iloc[i], df_app["Attendance"].iloc[i])
       for i in range(num_nodes)}
node_coords = np.array([pos[i] for i in range(num_nodes)])
mean_att    = df_app["Attendance"].mean()

# 5. Rocket palette for the network
rocket_colorscale = [
    [0.0,  "#03051A"], [0.15, "#310A5D"], [0.3,  "#6A176E"],
    [0.45, "#A42C60"], [0.6,  "#D3504A"], [0.75, "#F0855A"],
    [0.9,  "#F9C496"], [1.0,  "#FAEBDD"],
]

fig = go.Figure()

# Edges with colour gradient based on average attendance
edge_x, edge_y, edge_z, edge_colors = [], [], [], []
for u, v in G.edges():
    x0, y0, z0 = pos[u]
    x1, y1, z1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])
    avg_att = (df_app.loc[u, "Attendance"] + df_app.loc[v, "Attendance"]) / 2
    edge_colors.extend([avg_att, avg_att, avg_att])

fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z, mode="lines",
    line=dict(width=2, color=edge_colors, colorscale=rocket_colorscale, showscale=False),
    hoverinfo="none", showlegend=False,
))

# Cluster footprints (volume hull) plus node markers
for c_id, info in cluster_info.items():
    c_idx = df_app[df_app["CLUSTER"] == c_id].index
    pts   = node_coords[c_idx]

    if len(c_idx) >= 4:
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], alphahull=0.5, opacity=0.08,
            color=info["color"], name=f"Segment: {info['name']}", legendgroup=info["name"],
        ))

    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
        name=info["name"], legendgroup=info["name"],
        marker=dict(
            size=(df_app.loc[c_idx, "Attendance"] / mean_att) * 12,
            color=df_app.loc[c_idx, "Attendance"],
            colorscale=rocket_colorscale,
            showscale=(c_id == 0),
            colorbar=dict(
                title=dict(text="Attendance Yield", font=dict(color="white")),
                thickness=15, x=1.15, tickfont=dict(color="white"),
            ) if c_id == 0 else None,
            line=dict(color="white", width=0.5),
        ),
        text=[f"ID: {idx}<br>Att: {df_app.loc[idx, 'Attendance']}" for idx in c_idx],
        hoverinfo="text",
    ))

# Axis labels via dummy traces
for label in ["X: Temperature", "Y: Opponent Win %", "Z: Attendance Yield"]:
    fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers",
                               marker=dict(size=0, opacity=0), name=label, showlegend=True))

def axis_style(title):
    return dict(
        showbackground=True, backgroundcolor="black", showgrid=False, zeroline=False,
        showticklabels=False,
        title=dict(text=title, font=dict(color="rgba(255,255,255,0.5)", size=10)),
    )

fig.update_layout(
    title=dict(
        text=f"Strategic Asset Mapping: Systemic Risk Engine ({num_nodes} Obs.)",
        font=dict(size=22, color="white"), x=0.05,
    ),
    width=1400, height=900,
    paper_bgcolor="black", plot_bgcolor="black",
    margin=dict(l=0, r=0, b=0, t=60),
    scene=dict(
        xaxis=axis_style("Weather Environment (X)"),
        yaxis=axis_style("Opponent Quality (Y)"),
        zaxis=axis_style("Attendance Performance (Z)"),
        bgcolor="black",
    ),
    legend=dict(font=dict(color="white"), bgcolor="rgba(0,0,0,0)", y=0.9, x=0.01),
)

fig.show()


In [ ]:
# Feature engineering and OLS with VIF and residual plot
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_app = pd.read_csv(file_name)
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_app.columns:
        df_app[c] = pd.to_numeric(df_app[c], errors="coerce")
df_app = df_app.dropna(subset=["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]).reset_index(drop=True)

# Feature engineering
df_app["TEMP_SQ"]       = df_app["Temp"] ** 2
df_app["QUALITY_INDEX"] = df_app["Win%"] * df_app["OpWin%"]
df_app["PROMO_WEEKEND"] = df_app["Promotion"] * df_app["Weekend"]

# Feature pool for the model
features = ["Temp", "TEMP_SQ", "Win%", "OpWin%", "QUALITY_INDEX", "PROMO_WEEKEND"]
X = sm.add_constant(df_app[features])
y_app = df_app["Attendance"]

# Fit
model       = sm.OLS(y_app, X).fit()
predictions = model.predict(X)
resid_app   = model.resid

print(model.summary())

# VIF table
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"]     = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data)

# Residual plot
plt.figure(figsize=(10, 5))
sns.residplot(x=predictions, y=resid_app, lowess=True, line_kws={"color": "red", "lw": 1})
plt.title("Residuals vs Fitted Values (Best Model)")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()


## IX. Advanced Systemic Sensitivity Analysis  
### Polynomial Linearization and Best Subset Optimization

This analysis advances the *Systemic Risk Engine* by transitioning from a global linear assumption to a segmented, non-linear sensitivity framework. Following the professor's guidelines, we implemented:

- Feature Engineering  
- Best Subset Selection  
- Variance Inflation Factor (VIF) validation  

The objective is to refine the attendance forecasting model while preserving interpretability and statistical robustness.

---

## 1. Feature Engineering and Linearization Logic

To capture the "sweet spot" of attendance with respect to exogenous factors, we introduced transformed features designed to linearize non-linear economic effects within an OLS framework.

### Polynomial Transformation ($TEMP^2$)

Attendance does not react linearly to temperature. Both cold and extreme heat penalize attendance, generating a parabolic relationship.

Including $TEMP^2$ allows us to:
- Capture curvature effects
- Model diminishing marginal sensitivity
- Preserve linear estimation while incorporating non-linearity

---

### Interaction Term ($QUALITY\_INDEX = WIN\% \times OPWIN\%$)

This variable captures **Hype Synergy**:

- A matchup between two high-performing teams generates attendance greater than the sum of individual win-rate effects.
- The interaction term isolates this multiplicative premium effect.

---

### Relative Strength ($WIN\_DIFF = WIN\% - OPWIN\%$)

Measures competitiveness:

- Small difference → balanced game → higher uncertainty → potential attendance boost  
- Large difference → predictable outcome → lower engagement  

---

## 2. Model Selection: Best Subset and VIF Analysis

We evaluated all possible feature combinations ($2^k$ iterations) and selected the subset that maximizes Adjusted $R^2$ while maintaining parsimony.

### Variance Inflation Factor (VIF)

| Feature          | VIF   | Status            |
|------------------|-------|------------------|
| TEMP             | > 100 | High (Monitored) |
| TEMP²            | > 100 | High (Monitored) |
| WIN%             | ~1.1  | Stable           |
| OPWIN%           | ~1.0  | Stable           |
| QUALITY_INDEX    | ~2.5  | Stable           |

**Note:**  
Although $TEMP$ and $TEMP^2$ exhibit high VIF due to their deterministic relationship, they are retained because their joint inclusion materially reduces residual variance and improves overall model fit.

---

## 3. Visualization and Statistical Density

### Density Estimation (KDE)

The Kernel Density Estimation (KDE) approximates the continuous probability distribution of attendance.

Compared to a histogram:
- KDE applies a Gaussian kernel
- Produces a smooth density curve
- Identifies the "Operational Core" where most games cluster

This allows management to visualize the most probable revenue outcomes rather than discrete frequency bins.

---

### Residual Analysis (Fitted vs Residuals)

The residual plot of the Best Subset model shows:

- Random dispersion around zero
- No visible structural patterns
- Homoskedastic error distribution

The orange LOWESS smoothing line remains approximately flat, indicating that polynomial terms successfully captured systematic non-linear effects, leaving predominantly stochastic noise.

---

## 4. Regression Matrix: Linear vs Quadratic

A 4x3 regression grid compares:

- Linear fit (Blue)
- Quadratic fit (Orange)

### Climate Stress

Temperature often exhibits a pronounced quadratic curvature, confirming high sensitivity to weather extremes.

### Premium Engine

At higher attendance levels, the relationship with team quality ($WIN\%$) becomes more linear.  
Once a performance threshold is reached, marginal quality improvements generate more stable attendance effects.

---

### Conclusion

The integration of polynomial transformations, interaction effects, and best subset optimization materially strengthens the forecasting architecture by:

- Capturing non-linear economic dynamics  
- Preserving interpretability  
- Controlling multicollinearity  
- Enhancing predictive stability  

This framework provides a statistically rigorous and economically coherent extension of the original linear model.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import itertools

# Robust raw load
file_name = next((p for p in ["G4_excel_dataset.csv", "G4 excel dataset.csv"] if os.path.exists(p)),
                 "G4_excel_dataset.csv")
df_app = pd.read_csv(file_name)
for c in ["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]:
    if c in df_app.columns:
        df_app[c] = pd.to_numeric(df_app[c], errors="coerce")
df_app = df_app.dropna(subset=["Attendance", "Temp", "Win%", "OpWin%", "Weekend", "Promotion"]).reset_index(drop=True)

# 1. Feature engineering
df_app["TEMP_SQ"]       = df_app["Temp"] ** 2
df_app["QUALITY_INDEX"] = df_app["Win%"] * df_app["OpWin%"]
df_app["WIN_DIFF"]      = df_app["Win%"] - df_app["OpWin%"]

# 2. Clustering on core metrics
scaler    = StandardScaler()
df_scaled = scaler.fit_transform(df_app[["Attendance", "Temp", "Win%", "OpWin%"]])
kmeans    = KMeans(n_clusters=4, random_state=42, n_init=10)
df_app["CLUSTER"] = kmeans.fit_predict(df_scaled)
cluster_map = {0: "Climate Stress", 1: "Standard Baseline", 2: "Premium Engine", 3: "Efficiency Gap"}
df_app["CLUSTER_NAME"] = df_app["CLUSTER"].map(cluster_map)

# 3. Best subset selection on the engineered feature pool
def get_best_subset(y_vec, X_pool):
    best_adj_r2 = -np.inf
    best_model_  = None
    best_subset_ = None
    for k in range(1, len(X_pool.columns) + 1):
        for combo in itertools.combinations(X_pool.columns, k):
            X_sub = sm.add_constant(X_pool[list(combo)])
            m_    = sm.OLS(y_vec, X_sub).fit()
            if m_.rsquared_adj > best_adj_r2:
                best_adj_r2  = m_.rsquared_adj
                best_model_  = m_
                best_subset_ = list(combo)
    return best_model_, best_subset_

features_pool = df_app[["Temp", "TEMP_SQ", "Win%", "OpWin%", "QUALITY_INDEX",
                        "Weekend", "Promotion", "WIN_DIFF"]]
best_model_app, best_feats = get_best_subset(df_app["Attendance"], features_pool)

print("Best subset (Adj R-sq):", best_model_app.rsquared_adj.round(4))
print("Selected features:", best_feats)

# 4. VIF for the best subset
X_best_app = sm.add_constant(df_app[best_feats])
vif_df_app = pd.DataFrame()
vif_df_app["Feature"] = X_best_app.columns
vif_df_app["VIF"]     = [variance_inflation_factor(X_best_app.values, i) for i in range(X_best_app.shape[1])]
print("\n--- VIF (Best Subset) ---")
print(vif_df_app)

# 5. Visualisations
# 5a. KDE of attendance
plt.figure(figsize=(10, 6))
sns.kdeplot(df_app["Attendance"], fill=True, color="#A42C60")
plt.title("KDE: Attendance Result Density Estimation")
plt.tight_layout()
plt.show()

# 5b. Residual plot for the best subset
plt.figure(figsize=(10, 6))
sns.residplot(x=best_model_app.fittedvalues, y=best_model_app.resid, lowess=True,
              line_kws={"color": "orange", "lw": 2},
              scatter_kws={"alpha": 0.6, "color": "#310A5D"})
plt.title("Residuals vs Fitted (Best Subset Model)")
plt.tight_layout()
plt.show()

# 5c. Linear vs quadratic regplots, faceted by cluster
continuous_vars = ["Temp", "Win%", "OpWin%"]
clusters_order  = sorted(df_app["CLUSTER_NAME"].dropna().unique())

fig, axes = plt.subplots(len(clusters_order), len(continuous_vars), figsize=(18, 15))
# Handle 1xN edge case (not expected here, but defensive)
if len(clusters_order) == 1:
    axes = np.array([axes])

for i, cluster in enumerate(clusters_order):
    c_data = df_app[df_app["CLUSTER_NAME"] == cluster]
    for j, var in enumerate(continuous_vars):
        ax = axes[i, j]
        sns.regplot(data=c_data, x=var, y="Attendance", ax=ax, order=1, scatter=True,
                    line_kws={"color": "blue", "label": "Linear"},
                    scatter_kws={"alpha": 0.5, "color": "#A42C60"})
        sns.regplot(data=c_data, x=var, y="Attendance", ax=ax, order=2, scatter=False,
                    line_kws={"color": "orange", "label": "Quadratic", "linestyle": "--"})
        ax.set_title(f"{cluster}: {var}")

plt.tight_layout()
plt.show()
